# Malaysia Weather Data Analysis

This project explores 51,693 cleaned weather records from selected locations in Kuala Lumpur and Pulau Pinang. It focuses on data quality, location coverage, feature engineering, and how humidity and dew point support communication about outdoor comfort.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path("malaysia_weather_cleaned.csv")
weather = pd.read_csv(DATA_PATH)
print("Dataset shape:", weather.shape)
display(weather.head())

## 1. Data quality and coverage

The cleaned dataset retains Boolean flags showing which values were imputed. These flags make the cleaning process auditable and prevent filled values from being mistaken for direct observations.

In [ ]:
print("Duplicate rows:", weather.duplicated().sum())
print("Remaining missing values:", int(weather.isna().sum().sum()))

imputation_flags = [c for c in weather.columns if c.endswith("_was_imputed")]
imputation_summary = pd.DataFrame({
    "variable": [c.replace("_was_imputed", "") for c in imputation_flags],
    "imputed_count": [int(weather[c].sum()) for c in imputation_flags],
    "imputed_percent": [weather[c].mean() * 100 for c in imputation_flags],
}).sort_values("imputed_percent", ascending=False)
display(imputation_summary.round(2))

state_counts = weather["state"].value_counts()
display(state_counts.rename("records").to_frame())

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(x=state_counts.index, y=state_counts.values, color="#3977a8")
plt.title("Weather records by state")
plt.xlabel("State")
plt.ylabel("Records")
plt.tight_layout()
plt.show()

The data cover two states and are unevenly distributed, with more records from Kuala Lumpur. Results should therefore be interpreted as patterns in this dataset rather than a complete picture of Malaysia.

## 2. Feature engineering

The analysis derives variables that are easier to use in comparisons and public communication: datetime, temperature difference, gust difference, rain-event flags, pressure change, and UV categories.

In [ ]:
weather["datetime"] = pd.to_datetime(weather["datetime"], errors="coerce")
weather["temperature_difference"] = weather["temperature"] - weather["wind_chill"]
weather["gust_difference"] = weather["gust"] - weather["wind_speed"]
weather["rain_event"] = weather["precipitation_total"] > 0

uv_bins = [-np.inf, 2, 5, 7, 10, np.inf]
uv_labels = ["Low", "Moderate", "High", "Very High", "Extreme"]
weather["uv_category"] = pd.cut(weather["uv_index"], bins=uv_bins, labels=uv_labels)

display(weather[["datetime", "temperature_difference", "gust_difference", "rain_event", "uv_category"]].head())

## 3. Humidity and dew point

Humidity is familiar to the public, while dew point provides a more technical view of atmospheric moisture. Their relationship is examined through correlation, distributions, and monthly averages.

In [ ]:
comfort = weather[["humidity", "dew_point"]].dropna()
corr = comfort["humidity"].corr(comfort["dew_point"])
print(f"Pearson correlation: {corr:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.scatterplot(data=comfort.sample(min(5000, len(comfort)), random_state=42),
                x="humidity", y="dew_point", alpha=0.25, ax=axes[0])
axes[0].set_title("Humidity and dew point")

monthly = weather.groupby("month_number")[["humidity", "dew_point"]].mean()
monthly.plot(ax=axes[1], marker="o")
axes[1].set_title("Monthly average humidity and dew point")
axes[1].set_xlabel("Month")
plt.tight_layout()
plt.show()

## 4. Conclusion

The dataset supports using humidity as the primary public-facing indicator for outdoor comfort because it is widely understood. Dew point remains useful as a supporting technical measure. Temperature, precipitation, wind speed, gust, and UV categories should accompany comfort messages when communicating broader weather conditions.

The analysis is descriptive. It covers selected locations in Kuala Lumpur and Pulau Pinang, includes imputed values, and does not test public understanding directly.